In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [4]:
# SIMULATED DATASET

np.random.seed(42)

# SIMULATED PRODUCTION-LIKE EVENT DATASET

n= 20000
df = pd.DataFrame({
    "user_id": range(n),
    "segment": np.random.choice(["enterprise", "mid_market", "SMB"], n, p=[0.2,0.3,0.5]),
    "device": np.random.choice(["web", "mobile"], n),
    "region": np.random.choice(["US", "EU", "APAC"], n)
})

# Assign experiment groups (A = control, B = pricing change)
df["group"] = np.random.choice(["A", "B"], n)


In [5]:
# 2. BUSINESS BEHAVIOR SIMULATION (REALISTIC)

def conversion_rate(row):
    base = 0.10

    if row["segment"] == "enterprise":
        base += 0.10
    elif row["segment"] == "mid_market":
        base += 0.05

    if row["group"] == "B":
        base += 0.03  # pricing optimization uplift effect

    return np.random.binomial(1, base)

df["converted"] = df.apply(conversion_rate, axis=1)

# revenue logic (pricing experiment impact)
df["revenue"] = df["converted"] * np.where(
    df["group"] == "A",
    np.random.normal(100, 20, n),
    np.random.normal(115, 25, n)  # uplift in B
)

In [6]:
# 3. SQL-LIKE AGGREGATION LAYER

kpi_table = df.groupby("group").agg(
    conversion_rate=("converted", "mean"),
    avg_revenue=("revenue", "mean"),
    total_revenue=("revenue", "sum")
).reset_index()

print("\n=== KPI TABLE ===")
print(kpi_table)



=== KPI TABLE ===
  group  conversion_rate  avg_revenue  total_revenue
0     A         0.142313    14.274758  142433.535456
1     B         0.168429    19.388266  194309.203480


In [7]:
# 4. STATISTICAL TESTING

a = df[df["group"] == "A"]["converted"]
b = df[df["group"] == "B"]["converted"]

t_stat, p_value = stats.ttest_ind(a, b)

print("\nT-Statistic:", t_stat)
print("P-Value:", p_value)


T-Statistic: -5.100423453552735
P-Value: 3.4199014224017216e-07


In [8]:
# 5. REVENUE LIFT ANALYSIS

rev_a = df[df["group"] == "A"]["revenue"].mean()
rev_b = df[df["group"] == "B"]["revenue"].mean()

revenue_lift = ((rev_b - rev_a) / rev_a) * 100

print("\nRevenue A:", rev_a)
print("Revenue B:", rev_b)
print("Revenue Lift %:", revenue_lift)


Revenue A: 14.274758013261552
Revenue B: 19.388266162397002
Revenue Lift %: 35.82203035865752


In [9]:
# 6. SEGMENT LEVEL INSIGHTS

segment_analysis = df.groupby(["segment", "group"]).agg(
    conv=("converted", "mean"),
    rev=("revenue", "mean")
).reset_index()

print("\n=== SEGMENT ANALYSIS ===")
print(segment_analysis)


=== SEGMENT ANALYSIS ===
      segment group      conv        rev
0         SMB     A  0.109069  10.935681
1         SMB     B  0.132678  15.274622
2  enterprise     A  0.198020  19.753592
3  enterprise     B  0.246734  28.361574
4  mid_market     A  0.160569  16.188097
5  mid_market     B  0.175738  20.252952
